In [ ]:
data_cleaning.ipynb

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load existing sales dataset
df = pd.read_csv("../week1_day5/day5_sales.csv")

# Display first 5 rows
df.head()

,sale_id,sale_date,region,category,product,quantity,sales_amount,profit
0,1,2026-01-05,Karnataka,Electronics,Laptop,2,80000.0,10000.0
1,2,2026-01-10,Kerala,Furniture,Chair,5,15000.0,3000.0
2,3,2026-02-12,Tamil Nadu,Electronics,Mobile,3,45000.0,7000.0
3,4,2026-02-15,Karnataka,Clothing,Shirt,10,10000.0,2500.0
4,5,2026-03-05,Kerala,Electronics,Tablet,4,40000.0,6000.0


In [2]:
# Check dataset size
print("Dataset Shape:", df.shape)

# Check column names and data types
print("\nData Types:")
print(df.dtypes)

# Check missing values
print("\nMissing Values:")
print(df.isnull().sum())

# Check duplicate records
print("\nDuplicate Rows:", df.duplicated().sum())

Dataset Shape: (120, 8)

Data Types:
sale_id           int64
sale_date        object
region           object
category         object
product          object
quantity          int64
sales_amount    float64
profit          float64
dtype: object

Missing Values:
sale_id         0
sale_date       0
region          0
category        0
product         0
quantity        0
sales_amount    0
profit          0
dtype: int64

Duplicate Rows: 0


In [3]:
# Create a copy of the original dataset
cleaned_df = df.copy()

print("Original dataset rows:", len(df))
print("Working dataset rows:", len(cleaned_df))

Original dataset rows: 120
Working dataset rows: 120


In [4]:
# Count missing values
missing_values = cleaned_df.isnull().sum()

print("Missing values in each column:")
print(missing_values)

Missing values in each column:
sale_id         0
sale_date       0
region          0
category        0
product         0
quantity        0
sales_amount    0
profit          0
dtype: int64


In [5]:
# Fill missing numeric values with median
for col in cleaned_df.select_dtypes(include="number").columns:
    cleaned_df[col] = cleaned_df[col].fillna(cleaned_df[col].median())

print("Missing values after filling numeric columns:")
print(cleaned_df.isnull().sum())

Missing values after filling numeric columns:
sale_id         0
sale_date       0
region          0
category        0
product         0
quantity        0
sales_amount    0
profit          0
dtype: int64


In [6]:
# Fill missing text values with mode
for col in cleaned_df.select_dtypes(include="object").columns:
    if cleaned_df[col].isnull().sum() > 0:
        mode_value = cleaned_df[col].mode()

        if not mode_value.empty:
            cleaned_df[col] = cleaned_df[col].fillna(mode_value[0])

print("Missing values after mode imputation:")
print(cleaned_df.isnull().sum())

Missing values after mode imputation:
sale_id         0
sale_date       0
region          0
category        0
product         0
quantity        0
sales_amount    0
profit          0
dtype: int64


In [7]:
# Count duplicates before removal
print("Duplicates before cleaning:", cleaned_df.duplicated().sum())

# Remove duplicate rows
cleaned_df = cleaned_df.drop_duplicates()

# Check duplicates after removal
print("Duplicates after cleaning:", cleaned_df.duplicated().sum())

Duplicates before cleaning: 0
Duplicates after cleaning: 0


In [8]:
# Select numeric columns
numeric_cols = cleaned_df.select_dtypes(include="number").columns

# Detect outliers using IQR
for col in numeric_cols:
    Q1 = cleaned_df[col].quantile(0.25)
    Q3 = cleaned_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = cleaned_df[
        (cleaned_df[col] < lower_limit) |
        (cleaned_df[col] > upper_limit)
    ]

    print(f"{col}: {len(outliers)} outliers")

sale_id: 0 outliers
quantity: 12 outliers
sales_amount: 12 outliers
profit: 0 outliers


In [9]:
# Cap outliers using IQR method
for col in numeric_cols:
    Q1 = cleaned_df[col].quantile(0.25)
    Q3 = cleaned_df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    cleaned_df[col] = cleaned_df[col].clip(
        lower=lower_limit,
        upper=upper_limit
    )

print("Outlier capping completed!")

Outlier capping completed!


In [10]:
# Standardise text columns
text_cols = cleaned_df.select_dtypes(include="object").columns

for col in text_cols:
    cleaned_df[col] = (
        cleaned_df[col]
        .str.strip()
        .str.lower()
    )

print("Text standardisation completed!")

# Display the updated text values
for col in text_cols:
    print(f"\n{col}:")
    print(cleaned_df[col].unique()[:20])

Text standardisation completed!

sale_date:
['2026-01-05' '2026-01-10' '2026-02-12' '2026-02-15' '2026-03-05'
 '2026-03-10' '2026-03-15' '2026-04-02' '2026-04-08' '2026-04-12']

region:
['karnataka' 'kerala' 'tamil nadu']

category:
['electronics' 'furniture' 'clothing']

product:
['laptop' 'chair' 'mobile' 'shirt' 'tablet' 'table' 'jeans']


In [11]:
# Before cleaning
print("BEFORE CLEANING")
print("Rows:", df.shape[0])
print("Missing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())

# After cleaning
print("\nAFTER CLEANING")
print("Rows:", cleaned_df.shape[0])
print("Missing values:", cleaned_df.isnull().sum().sum())
print("Duplicate rows:", cleaned_df.duplicated().sum())

BEFORE CLEANING
Rows: 120
Missing values: 0
Duplicate rows: 0

AFTER CLEANING
Rows: 120
Missing values: 0
Duplicate rows: 0


In [12]:
# Save cleaned dataset
cleaned_df.to_csv("cleaned_dataset.csv", index=False)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!


In [13]:
cleaning_log = """
# Data Cleaning Log – Week 2 Day 4

## Cleaning Operations

1. Loaded the original sales dataset.
2. Created a copy to preserve the original data.
3. Identified missing values using isnull().sum().
4. Filled missing numeric values using median.
5. Filled missing text values using mode.
6. Removed duplicate records using drop_duplicates().
7. Detected outliers using the IQR method.
8. Capped numeric outliers using IQR limits.
9. Standardised text by removing extra spaces and converting to lowercase.

## Before and After Comparison

- Original rows: {}
- Cleaned rows: {}
- Missing values before: {}
- Missing values after: {}
- Duplicate rows before: {}
- Duplicate rows after: {}

## Note

Cleaning decisions should be reviewed to ensure that genuine
business values are not incorrectly changed.
""".format(
    len(df),
    len(cleaned_df),
    df.isnull().sum().sum(),
    cleaned_df.isnull().sum().sum(),
    df.duplicated().sum(),
    cleaned_df.duplicated().sum()
)

with open("cleaning_log.md", "w", encoding="utf-8") as file:
    file.write(cleaning_log)

print("Cleaning log created successfully!")

Cleaning log created successfully!


In [14]:
import os

print("Files in week2_day4 folder:")

for file in os.listdir("."):
    print(file)

Files in week2_day4 folder:
.ipynb_checkpoints
cleaned_dataset.csv
cleaning_log.md
Untitled.ipynb
Untitled1.ipynb
